# Tutorial 10 — MEA Neurotoxicity & Dose-Response Modeling
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Multi-Electrode Arrays (MEA) measure electrical activity of neuronal networks in culture. Changes in firing rate, burst frequency, and synchrony indicate neurotoxic effects. This approach was central to my BHSAI work assessing 200+ chemicals.

In [ ]:
!pip install pandas numpy matplotlib scipy scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

np.random.seed(42)

def hill_equation(c, bottom, top, ec50, hill):
    return bottom + (top - bottom) / (1 + (ec50/c)**hill)

def simulate_mea(name, ec50, hill=1.5, bottom=0.0, top=1.0, noise=0.08):
    concs = np.array([0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0])  # uM
    effect = hill_equation(concs, bottom, top, ec50, hill)
    effect += np.random.normal(0, noise, len(concs))
    effect = np.clip(effect, -0.1, 1.1)
    return pd.DataFrame({"chemical": name, "concentration_uM": concs, "effect": effect})

chemicals = [
    ("Chlorpyrifos",  0.08),
    ("Deltamethrin",  0.3),
    ("Rotenone",      0.05),
    ("Bisphenol A",   2.5),
    ("PBDE-47",       1.2),
    ("Lindane",       0.9),
]

dfs = [simulate_mea(n, ec) for n, ec in chemicals]
data = pd.concat(dfs, ignore_index=True)
print(data.groupby("chemical").agg({"concentration_uM": ["min","max"], "effect": ["min","max"]}).round(3))

## Fit dose-response curves

In [ ]:
results = []
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for ax, (name, true_ec50) in zip(axes.flatten(), chemicals):
    sub = data[data["chemical"]==name]
    x, y = sub["concentration_uM"].values, sub["effect"].values

    try:
        popt, _ = curve_fit(hill_equation, x, y,
                            p0=[0, 1, true_ec50, 1.5],
                            bounds=([-.2,0.5,1e-4,0.5],[.2,1.5,100,5]),
                            maxfev=5000)
        bottom, top, ec50_fit, hill_fit = popt
        x_fit = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
        y_fit = hill_equation(x_fit, *popt)
        ax.plot(x_fit, y_fit, color="#1565c0", lw=2.5, label=f"EC50={ec50_fit:.3f} μM")
        results.append({"chemical": name, "EC50_uM": round(ec50_fit, 4),
                        "Hill": round(hill_fit, 2), "Top": round(top, 3)})
    except Exception as e:
        results.append({"chemical": name, "EC50_uM": None, "Hill": None, "Top": None})

    ax.scatter(x, y, color="#e65100", zorder=5, s=50, label="Observed")
    ax.set_xscale("log"); ax.set_xlim([x.min()*0.5, x.max()*2])
    ax.axhline(0.5, color="gray", linestyle="--", lw=0.8, alpha=0.7)
    ax.set_title(name, fontsize=10); ax.set_xlabel("Concentration (μM)"); ax.set_ylabel("Effect (normalized)")
    ax.legend(fontsize=8)

plt.suptitle("MEA Dose-Response Curves — Neurotoxicity Panel", fontsize=12, y=1.01)
plt.tight_layout(); plt.savefig("mea_dose_response.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
res_df = pd.DataFrame(results).sort_values("EC50_uM")
print("\nFitted EC50 values (lower = more potent):")
print(res_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7,4))
colors = ["#c0392b" if v<0.1 else "#e67e22" if v<1 else "#27ae60"
          for v in res_df["EC50_uM"].fillna(99)]
bars = ax.barh(res_df["chemical"], np.log10(1/res_df["EC50_uM"].fillna(0.01)),
               color=colors, height=0.6)
ax.set_xlabel("log10(1/EC50)  →  more potent")
ax.set_title("Neurotoxicity Potency Ranking")
for bar, ec in zip(bars, res_df["EC50_uM"]):
    ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
            f"{ec:.3f} μM" if ec else "n/a", va="center", fontsize=9)
plt.tight_layout(); plt.savefig("potency_rank.png", dpi=150); plt.show()

## Key takeaways
- Hill equation: Effect = Bottom + (Top - Bottom)/(1 + (EC50/C)^Hill)
- EC50 is the concentration producing 50% of maximal effect — the key potency metric
- MEA provides 8+ endpoints per well (MFR, burst rate, synchrony, etc.) — we modeled just effect here
- Chlorpyrifos (EC50 ~0.08 μM) is the most potent in this panel — consistent with its mechanism (AChE inhibition)